# Official fast.ai course22 original 02 — Colab adaptation

Original: [02-saving-a-basic-fastai-model.ipynb](https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/clean/02-saving-a-basic-fastai-model.ipynb)

Pinned source commit: `230390584ba5b990e20487d68681a9ec5b258eff`. Source SHA256: `5e0a8b7dff2bbbb133a44ee3bc366bdb1ffdf262aa06d30bf6c9f940ae19bbe3`.

This additional notebook preserves the full original computational experiments. Existing independently written companions 00–07 remain separate. Set QUICK_RUN=False; run every code cell. Source notebooks and authorship belong to Jeremy Howard and fast.ai. No separate course22 repository license was found at the pinned commit; no fastbook license is applied to these files.

- Preserved full Oxford-IIIT Pets cat/dog training, .2 split, seed42, resize192, ResNet18 and fine_tune(3). Export path isolated and a trusted local reload demonstration added. No public deployment.

Public publication and competition submission have not been performed. Saved predictions are learning artifacts; the 418 Titanic test targets are unavailable. Original conceptual prompts/headings are retained below.


In [1]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '19_original_02_save_pet_model.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)
if IN_COLAB:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime for this full image-training exercise.')

Exercise folder: /content/HaiLong_Colab_Exercises/03_fastai
Run mode: FULL EXERCISE
Results: /content/drive/MyDrive/HaiLong_Colab_Results/03_fastai/results


In [2]:
# Full-source execution and durable evidence helpers (Colab adaptation).
import hashlib, json, shutil, urllib.request
from datetime import datetime, timezone
if QUICK_RUN or os.environ.get('COURSE_SMOKE') == '1':
    raise ValueError('These original notebooks require full mode: set QUICK_RUN=False.')
RESULTS = COLAB_CONTEXT['folder'] / 'results' / Path(COLAB_CONTEXT['run']['notebook']).stem
MODELS = COLAB_CONTEXT['folder'] / 'models' / Path(COLAB_CONTEXT['run']['notebook']).stem
WORK = COLAB_CONTEXT['folder'] / 'data' / Path(COLAB_CONTEXT['run']['notebook']).stem
for directory in [RESULTS, MODELS, WORK]: directory.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def record_stage(stage, **evidence):
    target = RESULTS / 'stage_evidence.json'
    started = COLAB_CONTEXT['run']['started_utc']
    data = json.loads(target.read_text(encoding='utf-8')) if target.is_file() else {}
    if data.get('run_started_utc') != started:
        data = {'notebook': COLAB_CONTEXT['run']['notebook'], 'run_started_utc': started, 'stages': {}}
    data['stages'][stage] = {'recorded_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    data.update(quick_run=False, status='observed_stage_outputs', provider_course_completion_claimed=False,
                learner_personal_reflection_complete=False)
    temporary = target.with_suffix('.writing')
    temporary.write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')
    temporary.replace(target)

def verified_download(url, target, sha256):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file():
        temporary = target.with_name(target.name + '.download')
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
            raise ValueError('Source download checksum mismatch: ' + url)
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != sha256:
        raise ValueError('Existing source file differs from the pinned original: ' + str(target))
    return target

def save_plot(figure, name):
    if hasattr(figure, 'write_html'):
        figure.write_html(RESULTS / (name + '.html'), include_plotlyjs=True)
    elif hasattr(figure, 'savefig'):
        figure.savefig(RESULTS / (name + '.png'), dpi=150, bbox_inches='tight')
    return figure

record_stage('full_mode_setup', execution_location='hosted_colab' if IN_COLAB else 'local_validation',
             original_exercises_verified_complete=False)

## Evidence required for the original exercises

- Execute every nonempty original code cell top to bottom in hosted Colab; retain outputs and execution records.
- QUICK_RUN=False. Original experiment data coverage, epochs and learning rates are preserved.
- Original source attribution and the per-cell provenance map distinguish original exercises from companions.
- Save full notebook outputs and actual stage/checkpoint/artifact evidence in Drive.
- No provider badge, external publishing, competition submission or personal authorship is certified by execution.
- Download all Oxford-IIIT Pets images, cat/dog labels from filenames, valid_pct .2 seed42, resize192, pretrained ResNet18 fine_tune(3), export and reload the trained model. Public deployment is separate.


## Saving a Cats v Dogs Model

In [3]:
# fastai==2.8.12 is supplied by the checked setup profile.

In [4]:
from fastai.vision.all import *

In [5]:
path = untar_data(URLs.PETS)/'images'

<div><progress max="811706944" value="811712512"></progress> 100.00% [811712512/811706944 01:30&lt;00:00]</div>

In [6]:
def is_cat(x): return x[0].isupper() 

In [7]:
dls = ImageDataLoaders.from_name_func('.',
    get_image_files(path), valid_pct=0.2, seed=42,
    label_func=is_cat,
    item_tfms=Resize(192))

record_stage('all_pets_split', training_rows=len(dls.train_ds), validation_rows=len(dls.valid_ds), seed=42, validation_fraction=.2, image_size=192)

In [8]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

record_stage('pets_resnet18_full_fine_tune', requested_fine_tune_epochs=3, freeze_epochs=1, validation_metrics=[float(v) for v in learn.validate()])

Storage check (before_fit): artifacts 1.418 GB; configured budget 9.000 GB; additional estimate 0.194 GB; runtime free 71.036 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.418 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 71.036 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,error_rate,time
0,0.197715,0.041677,0.015562,00:40


Storage check (checkpoint_write): artifacts 1.465 GB; configured budget 9.000 GB; additional estimate 0.062 GB; runtime free 70.988 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.519 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 70.935 GB. Budget is not Drive quota.


...

epoch,train_loss,valid_loss,error_rate,time
0,0.073861,0.047239,0.012855,00:41
1,0.041450,0.026759,0.007442,00:43
2,0.020090,0.023156,0.006766,00:45


Storage check (checkpoint_write): artifacts 1.514 GB; configured budget 9.000 GB; additional estimate 0.196 GB; runtime free 70.943 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.702 GB; configured budget 9.000 GB; additional estimate 0.196 GB; runtime free 70.755 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.702 GB; configured budget 9.000 GB; additional estimate 0.196 GB; runtime free 70.752 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.702 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 70.752 GB. Budget is not Drive quota.


In [9]:
learn.export(MODELS / 'model.pkl')
# Additional local verification of the deployment artifact; no external publication.
reloaded = load_learner(MODELS / 'model.pkl')
example = dls.valid.items[0]
prediction = reloaded.predict(PILImage.create(example))
print('Reloaded model prediction:', prediction)
(RESULTS / 'deployment_README.txt').write_text('Original Pets model exported for local inference. Publishing to a public Space/site has not been performed.', encoding='utf-8')
record_stage('export_and_reload', exported_model=str(MODELS / 'model.pkl'), validation_example=str(example),
             prediction=str(prediction), public_deployment_performed=False)

/usr/local/lib/python3.12/dist-packages/fastai/learner.py:456: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


Reloaded model prediction: ('True', tensor(1), tensor([0.2283, 0.7717]))


In [10]:
record_stage('reached_final_cell', all_previous_cells_independently_verified=False)
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [11]:
# Added by the course runner: inspect the executed runtime, not coursework.
import os as _hailong_os, json as _hailong_json, platform as _hailong_platform
_hailong_evidence = {
    'python': _hailong_platform.python_version(),
    'quick_run': bool(globals().get('QUICK_RUN', False)),
    'course_smoke': _hailong_os.environ.get('COURSE_SMOKE', '0'),
    'keras_backend': _hailong_os.environ.get('KERAS_BACKEND'),
    'legacy_keras': _hailong_os.environ.get('TF_USE_LEGACY_KERAS'),
    'base_model_commit': getattr(getattr(globals().get('model'), 'config', None), '_commit_hash', None),
}
_hailong_context = globals().get('COLAB_CONTEXT')
if _hailong_context and _hailong_context['folder'].relative_to(_hailong_context['root']).as_posix().startswith(('03_fastai', '04_fastai_part2')):
    from course_persistence import collect_reuse_evidence as _hailong_collect
    _hailong_evidence['fastai_reuse_evidence'] = _hailong_context['run'].get('reuse_evidence') or _hailong_collect(_hailong_context)
print('__HAILONG_RUNTIME_EVIDENCE__' + _hailong_json.dumps(_hailong_evidence))


__HAILONG_RUNTIME_EVIDENCE__{"python": "3.12.13", "quick_run": false, "course_smoke": "0", "keras_backend": null, "legacy_keras": null, "base_model_commit": null, "fastai_reuse_evidence": {"contract_version": 1, "notebook": "03_fastai/19_original_02_save_pet_model.ipynb", "source_sha256": "f9f4af55eb9ace211296394d62ae13d14790cc5ab1a0eb48435ea0b4c6bcf99d", "helper_sha256": {"03_fastai/convolution_workbook_exercise.py": "ecacb93aef6846c82d6a9eebc4809e1a6766410f04783ac132f247935af3a8fc", "03_fastai/fastai_utils.py": "a03e913e1c07f55aedd56ee6128096f3f952f4f8be22f83c4a22002589c4bd18", "03_fastai/wearable_objective_extension.py": "f9b0a8bf81a5bab8c10aee5e8795e9cef3df58e40925f7705e58301b18b3fbe9", "03_fastai/wearable_utils.py": "6de4ee45118d96e024bd4e0f3fd3552905b38a1ff01c0a5146578bb0d8263db9", "colab_runtime.py": "e11bedb13e1bd71a71c01cc49e558d05c89b057259fe52e64301d0c4d626a140", "course_persistence.py": "f14f7282d4b1e718b01858a61a2dca8b60f5135ba9971fc7923c50bf12cca3d2"}, "persistent_course_